In [ ]:
import pandas as pd
import numpy as np
import string 
from nltk.corpus import stopwords 
import nltk
from sklearn.feature_extraction.text import TfidfVectorizer 
from nltk.stem import SnowballStemmer
import re

In [ ]:
#Reading in all sheets
outcomes = pd.read_csv(r"data\outcomes.csv")
donations = pd.read_csv(r"data\donations.csv")
essays = pd.read_csv(r"data\essays.csv")
resources = pd.read_csv(r"data\resources.csv")
projects = pd.read_csv(r"data\projects.csv")


In [ ]:
#Just filtering for the project ID and final classification
outcomes_filtered = outcomes[['projectid', 'is_exciting']]

In [ ]:
#Merge all projects on projectid
final_df = outcomes_filtered.merge(donations, on = "projectid", how = 'inner').merge(essays, on = 'projectid', how = 'inner').merge(resources, on = 'projectid', how = 'inner').merge(projects, on ='projectid', how='inner')

In [ ]:
#Randomly sampling to only get one row per project
sampled_df = ( final_df.groupby('projectid') .apply(lambda g: g.sample(1, random_state=7506)) .reset_index(drop=True) )


In [ ]:
#Trying to keep all the exciting project for a class balance
exciting = sampled_df[sampled_df['is_exciting'] == 't']
remaining_needed = 100000- len(exciting)
non_exciting = sampled_df[sampled_df['is_exciting'] == 'f'] 
fill_rows = non_exciting.sample( n=remaining_needed, random_state=7506 )
df_sampled = pd.concat([exciting, fill_rows], ignore_index=True)

In [ ]:
#Removing identifier columns
df_sampled = df_sampled.drop(columns = ['teacher_acctid_x', 
'resourceid',  
'item_name', 
'item_number', 
'teacher_acctid_y',
'schoolid',
'school_ncesid',
'donor_acctid',
'donationid',
'vendorid',

#Removing categorical columns with too many categories
'donor_city',
'donor_zip',
'vendor_name',
'school_city',
'school_zip',
'school_district',
'school_county',

#Removing datetime columns as we cannot train on future dates 
'donation_timestamp',
'date_posted'
])




In [ ]:
#Making true/false a numeric boolean
df_sampled = df_sampled.replace('f', 0)
df_sampled = df_sampled.replace('t', 1)

In [ ]:
#Defining a text cleaner for text columns

nltk.download('stopwords') 
stop_words = set(stopwords.words('english'))
stemmer = SnowballStemmer("english")

def clean_text(text):
    if not isinstance(text, str):
        return text
    
    # remove real and escaped newlines
    text = re.sub(r'\r\n|\r|\n', ' ', text)
    text = re.sub(r'\\r\\n|\\n|\\r', ' ', text)

    # remove numbers 
    text = re.sub(r'\d+', ' ', text)
    
    # remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # lowercase
    text = text.lower()
    
    # remove stopwords and stem
    words = [ stemmer.stem(w) for w in text.split() if w not in stop_words ]
    
    return " ".join(words)


In [ ]:
#Cleaning text columns
text_columns = ['donation_message', 'title', 'short_description', 'need_statement', 'essay']
text_cols = text_columns  

df_sampled[text_columns] = df_sampled[text_columns].apply(
    lambda col: col.apply(clean_text)
)



In [ ]:
#Replacing this phrase with whitespace because it appears in almost every project
df_sampled['need_statement'] = df_sampled['need_statement'].str.replace('student need', '')
df_sampled[text_cols] = df_sampled[text_cols].fillna("NaN")

In [ ]:
#Filling NA values with "NaN" strings for tf-idf
df_sampled[['school_metro', 'donor_state', 'secondary_focus_subject','secondary_focus_area']] = df_sampled[['school_metro', 'donor_state','secondary_focus_subject','secondary_focus_area']].fillna('NaN')



In [ ]:
#Transforming the test into TD-IDF arrys 

essay_tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=25
)
essay_ = essay_tfidf.fit_transform(df_sampled['essay'])

donation_tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=10
)
donation_ = donation_tfidf.fit_transform(df_sampled['donation_message'])

title_tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=5
)
title_ = title_tfidf.fit_transform(df_sampled['title'])

desc_tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=10
)
desc_ = desc_tfidf.fit_transform(df_sampled['short_description'])

need_tfidf = TfidfVectorizer(
    stop_words='english',
    max_features=10
)
need_ = need_tfidf.fit_transform(df_sampled['need_statement'])



#Convert TF-IDF matrices to dataframes

#essay
essay_tfidf_df = pd.DataFrame(
    essay_.toarray(),
    columns=essay_tfidf.get_feature_names_out()
)

#donation message
donation_tfidf_df = pd.DataFrame(
    donation_.toarray(),
    columns=donation_tfidf.get_feature_names_out()
)


#title
title_tfidf_df = pd.DataFrame(
    title_.toarray(),
    columns=title_tfidf.get_feature_names_out()
)

#Short description
desc_tfidf_df = pd.DataFrame(
    desc_.toarray(),
    columns=desc_tfidf.get_feature_names_out()
)

#need statement
need_tfidf_df = pd.DataFrame(
    need_.toarray(),
    columns=need_tfidf.get_feature_names_out()
)



In [ ]:
#Adding source suffix to variables
essay_tfidf_df = essay_tfidf_df.add_suffix('_essay')
donation_tfidf_df = donation_tfidf_df.add_suffix('_donation')
title_tfidf_df = title_tfidf_df.add_suffix('_title')
desc_tfidf_df = desc_tfidf_df.add_suffix('_desc')
need_tfidf_df = need_tfidf_df.add_suffix('_need')


In [ ]:
#Readd in tf_idf dataframes to final dataframe
df_final = pd.concat( [df_sampled.reset_index(drop=True), essay_tfidf_df, donation_tfidf_df, title_tfidf_df, desc_tfidf_df, need_tfidf_df], axis=1 )
df_final.drop(columns = ['essay', 'donation_message', 'title', 'short_description', 'need_statement'], inplace = True)
df_final.shape

In [ ]:
#One hot encoding categorical variables
df_final= pd.get_dummies(df_final, columns=['donor_state', 
                                                 'payment_method', 
                                                 'project_resource_type', 
                                                 'school_state', 
                                                 'school_metro', 
                                                 'primary_focus_subject', 
                                                 'primary_focus_area',
                                                 'secondary_focus_subject',
                                                 'secondary_focus_area',
                                                 'resource_type',
                                                 'poverty_level',
                                                 'grade_level',
                                                 'teacher_prefix'])

df_final = df_final.replace('FALSE', 0).replace('TRUE', 1)

In [ ]:
#Reading out final CSV 
df_final.to_csv("output/final_data.csv")